# 🔗 Vinculador Manual de Órdenes de Compra
Usa este notebook para adjuntar una Orden de Compra a cualquier factura de manera manual.

### Instrucciones:
1. Pega la ruta completa de la factura en la variable `RUTA_FACTURA`.
2. Escribe el número de la Orden de Compra en la variable `NRO_OC`.
3. Ejecuta la celda.

In [4]:
import os
import fitz  # PyMuPDF
from oc_manager import buscar_pdf_oc
import shutil

# =======================================================================
# ⚙️ CONFIGURACIÓN MANUAL
# =======================================================================
RUTA_FACTURA = r"\\10.10.10.210\AyF_Trabajoadistancia\Compras\fc a subir\(AGOSTO) TELLO ALEJANDRO MATIAS FC 114.pdf"
NRO_OC = "21097"
# =======================================================================

def vincular_oc_manual(ruta_factura, nro_oc):
    if not os.path.exists(ruta_factura):
        print(f"❌ ERROR: No se encontró la factura en:\n   {ruta_factura}")
        return
        
    ruta_oc = buscar_pdf_oc(nro_oc)
    if not ruta_oc:
        print(f"❌ ERROR: No se encontró el PDF para la OC {nro_oc} en la red.")
        return
        
    print(f"✅ Factura encontrada: {os.path.basename(ruta_factura)}")
    print(f"✅ OC encontrada: {os.path.basename(ruta_oc)}")
    print("🔄 Fusionando documentos...")
    
    try:
        # Preparar nombre final
        dir_name = os.path.dirname(ruta_factura)
        base_name, ext = os.path.splitext(os.path.basename(ruta_factura))
        
        # Si ya tiene -OC, no lo duplicamos (o lo reemplazamos)
        if "-OC" in base_name:
            base_name = base_name.split("-OC")[0].strip()
            
        nuevo_nombre = f"{base_name} -OC {nro_oc}{ext}"
        ruta_final = os.path.join(dir_name, nuevo_nombre)
        
        # Fusionar PDFs
        doc_final = fitz.open(ruta_factura)
        doc_oc = fitz.open(ruta_oc)
        doc_final.insert_pdf(doc_oc)
        
        # Guardar en temporal primero para evitar errores si sobreescribimos el mismo archivo abierto
        ruta_temp = ruta_final + ".tmp"
        doc_final.save(ruta_temp)
        doc_final.close()
        doc_oc.close()
        
        # Reemplazar archivo viejo si el nombre cambia, o sobreescribir si es el mismo
        if ruta_factura != ruta_final:
            os.remove(ruta_factura)
        shutil.move(ruta_temp, ruta_final)
        
        print(f"🎉 ¡Éxito! Archivo guardado como:\n   {nuevo_nombre}")
        
    except Exception as e:
        print(f"❌ ERROR al fusionar: {e}")

# Ejecutar la función
vincular_oc_manual(RUTA_FACTURA, str(NRO_OC).strip())


✅ Factura encontrada: (AGOSTO) TELLO ALEJANDRO MATIAS FC 114.pdf
✅ OC encontrada: 21097.pdf
🔄 Fusionando documentos...
🎉 ¡Éxito! Archivo guardado como:
   (AGOSTO) TELLO ALEJANDRO MATIAS FC 114 -OC 21097.pdf
